# Terafab Decision Twin Colab Dashboard

Upload or edit a scenario JSON, run the model, inspect gates, and export a report.


In [5]:
# Install the package from the official Knowdyn repository
!pip install git+https://KNOWDYN@github.com/knowdyn/terafab-decision-twin.git

from terafab_decision_twin.schema import load_scenario, validate_scenario
from terafab_decision_twin.engine import run_scenario
from terafab_decision_twin.report import markdown_report

  Cloning https://****@github.com/knowdyn/terafab-decision-twin.git to /tmp/pip-req-build-8no1xfle
  Running command git clone --filter=blob:none --quiet 'https://****@github.com/knowdyn/terafab-decision-twin.git' /tmp/pip-req-build-8no1xfle
  Resolved https://****@github.com/knowdyn/terafab-decision-twin.git to commit 37f4a022cbd314dcc124a53d8e325bd71a21e377
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for terafab-decision-twin: filename=terafab_decision_twin-0.1.0-py3-none-any.whl size=24379 sha256=fa43a5323e47b05d2b0e7d0c19c4f4969ca0c4c87b23d5175e0ea9e194b8ea8a
  Stored in directory: /tmp/pip-ephem-wheel-cache-ar4dgq1u/wheels/0e/7e/9a/043b1d7f88e592afa7a922a47d7589859435af5e01871d4b57
Successfully built terafab-decision-twin


In [1]:
import os
import json

# Create the scenarios directory
os.makedirs('scenarios', exist_ok=True)

# Create a sample scenario file for demonstration
sample_scenario = {
    "name": "Baseline 2026",
    "version": "1.0",
    "parameters": {
        "throughput_target": 1000,
        "efficiency_threshold": 0.85
    },
    "gates": [
        {"name": "Capacity Check", "logic": "throughput > 800"},
        {"name": "Margin Check", "logic": "margin > 0.1"}
    ]
}

with open('scenarios/baseline_2026.json', 'w') as f:
    json.dump(sample_scenario, f, indent=4)

print("Sample scenario created at scenarios/baseline_2026.json")

Sample scenario created at scenarios/baseline_2026.json


In [9]:
from terafab_decision_twin.schema import load_scenario, validate_scenario
import json

# Adjusting the structure: renamed 'phase' to 'terafab_phase' and ensuring it is a dictionary
valid_scenario = {
    "metadata": {
        "scenario_id": "SCEN-001",
        "title": "Baseline 2026",
        "version": "1.0"
    },
    "time": {
        "start_year": 2026,
        "end_year": 2030,
        "time_step": "annual"
    },
    "terafab_phase": {"phase": "operational"},
    "energy": {
        "source": "grid",
        "site_electric_load_MW": 50.0,
        "load_factor": 0.9,
        "firm_capacity_MW": 60.0
    },
    "cooling": {
        "type": "liquid",
        "heat_rejection_capacity_MW": 5.0,
        "heat_rejection_fraction": 0.8,
        "cop": 3.5
    },
    "water": {
        "withdrawal_m3_per_MWh": 0.5,
        "consumptive_fraction": 0.2,
        "permit_withdrawal_m3_per_day": 500.0
    },
    "manufacturing": {
        "wafer_starts_per_month": 10000,
        "die_per_wafer": 500,
        "baseline_yield": 0.95,
        "throughput_target": 1000
    },
    "economics": {
        "currency": "USD",
        "electricity_price_USD_per_MWh": 60.0,
        "water_price_USD_per_m3": 1.5,
        "capex_USD": 1000000.0
    },
    "governance": {
        "framework": "standard",
        "partner_count": 3,
        "governance_complexity_index": 0.5
    },
    "policy": {
        "jobs_created": 200,
        "domestic_supply_security_index": 0.8,
        "public_legitimacy_index": 0.9,
        "compliance": ["ISO-14001"]
    },
    "control": {
        "automation_level": 4
    },
    "gates": [
        {"name": "Throughput Check", "logic": "manufacturing.wafer_starts_per_month > 5000"}
    ]
}

scenario_path = 'scenarios/baseline_2026.json'
with open(scenario_path, 'w') as f:
    json.dump(valid_scenario, f, indent=4)

scenario = load_scenario(scenario_path)
errors = validate_scenario(scenario)
print(f"Validation Errors: {errors}")
if not errors: print("Success: Scenario is now valid and compatible with engine.")
scenario

Validation Errors: []
Success: Scenario is now valid and compatible with engine.


{'metadata': {'scenario_id': 'SCEN-001',
  'title': 'Baseline 2026',
  'version': '1.0'},
 'time': {'start_year': 2026, 'end_year': 2030, 'time_step': 'annual'},
 'terafab_phase': {'phase': 'operational'},
 'energy': {'source': 'grid',
  'site_electric_load_MW': 50.0,
  'load_factor': 0.9,
  'firm_capacity_MW': 60.0},
 'cooling': {'type': 'liquid',
  'heat_rejection_capacity_MW': 5.0,
  'heat_rejection_fraction': 0.8,
  'cop': 3.5},
 'water': {'withdrawal_m3_per_MWh': 0.5,
  'consumptive_fraction': 0.2,
  'permit_withdrawal_m3_per_day': 500.0},
 'manufacturing': {'wafer_starts_per_month': 10000,
  'die_per_wafer': 500,
  'baseline_yield': 0.95,
  'throughput_target': 1000},
 'economics': {'currency': 'USD',
  'electricity_price_USD_per_MWh': 60.0,
  'water_price_USD_per_m3': 1.5,
  'capex_USD': 1000000.0},
 'governance': {'framework': 'standard',
  'partner_count': 3,
  'governance_complexity_index': 0.5},
 'policy': {'jobs_created': 200,
  'domestic_supply_security_index': 0.8,
  'pub

In [10]:
from terafab_decision_twin.engine import run_scenario

# Running the simulation with the validated scenario
result = run_scenario(scenario)
result['summary']

{'scenario_id': 'SCEN-001',
 'time_steps': 5,
 'energy_MWh': 1971000.0,
 'average_site_load_MW': 50.0,
 'peak_site_load_MW': 50.0,
 'minimum_firm_capacity_margin_MW': 2.500000000000007,
 'heat_rejection_required_MW': 40.0,
 'minimum_heat_rejection_margin_MW': -35.0,
 'cooling_auxiliary_energy_MWh': 450514.28571428574,
 'entropy_generation_MW_per_K': 0.007176530403094228,
 'exergy_destroyed_MW': 2.139682539682544,
 'average_exergy_efficiency': 0.0,
 'water_withdrawal_m3': 985500.0,
 'water_consumptive_use_m3': 197100.0,
 'wastewater_m3': 788400.0,
 'minimum_water_withdrawal_margin_m3_per_day': -40.0,
 'minimum_wastewater_discharge_margin_m3_per_day': 68.0,
 'good_die': 285000000.0,
 'compute_output_proxy_W': 0.0,
 'average_effective_yield': 0.95,
 'average_readiness_index': 0.0,
 'annualized_capex_USD': 101852.20882315059,
 'total_opex_USD': 119738250.0,
 'total_cost_USD': 120247511.04411575,
 'cost_per_good_die_USD': 0.42192109138286227,
 'cost_per_compute_watt_USD': None,
 'emissions_

In [ ]:
for gate in result['gates']:
    print(('PASS' if gate['passed'] else 'FAIL'), gate['name'], gate['margin'])


In [12]:
from terafab_decision_twin.report import markdown_report
from IPython.display import Markdown

report = markdown_report(result)
Markdown(report)

# Terafab Decision Twin Report — Baseline 2026

## Executive status

- Scenario ID: `SCEN-001`
- Overall gate result: **FAIL**
- Recommended phase action: `hold_or_redesign`

## Key scalar outputs

| Metric | Value | Unit |
|---|---:|---|
| Energy | 1.97e+06 | MWh |
| Peak site load | 50 | MW |
| Heat rejection required | 40 | MW |
| Water withdrawal | 9.855e+05 | m3 |
| Good die | 2.85e+08 | die |
| Average effective yield | 0.95 | fraction |
| Total cost | 1.2e+08 | USD |
| Cost per good die | 0.4219 | USD/die |
| Emissions | 0 | tCO2 |
| Public benefit | 0.714 | index |
| Public burden | 0 | index |
| Legitimacy margin | 0.714 | index |

## Due-diligence gates

| Gate | Status | Severity | Margin | Message |
|---|---|---|---:|---|
| dimensional | PASS | error | n/a | Key extensive outputs must be non-negative. |
| thermodynamic_heat_rejection | FAIL | error | -35 | Heat-rejection capacity must cover first-law heat load. |
| power_firm_capacity | PASS | error | 2.5 | Firm capacity must cover site load plus reserve margin. |
| water_withdrawal_permit | FAIL | error | -40 | Daily withdrawal must remain within permit capacity. |
| wastewater_discharge_permit | PASS | warning | 68 | Daily wastewater discharge should remain within permit capacity. |
| manufacturing_yield | PASS | error | 0.95 | Effective yield must be finite and within (0,1]. |
| economic_finiteness | PASS | error | 0.4219 | Cost per good die must be finite and non-negative. |
| policy_legitimacy | PASS | warning | 0.714 | Public burden should not overwhelm modeled benefit. |
| governance_risk | PASS | warning | 0.495 | Governance complexity should remain below high-risk threshold. |
| evidence | PASS | error | n/a | Evidence status audit passed. |

## Evidence audit

Status counts:

## Reproducibility

- `scenario_sha256`: `ff468928683de97efdd48d3a8f5102c4a4354c62218d0ce69062012aa5a4af6c`
- `result_sha256`: `0c00fb24291a93b509ea2b85cf5396aac2362633e370fbc01e0950788b5d6a94`

This report is a scenario-dependent model output, not a verified Terafab operating fact.
